In [38]:
import spacy
import json
from spacy import displacy
from scrapy.selector import Selector
from spacy.pipeline.senter import DEFAULT_SENTER_MODEL
import dateparser

In [39]:
articles = []
with open('/home/math2001/good-posts.json') as fp:
    for i, line in enumerate(fp):
        if i >= 100:
            break
        articles.append(json.loads(line))

nlp = spacy.load('en_core_web_sm')

with open('disease_list.json') as fp:
    DISEASES = [d['name'].lower() for d in json.load(fp)]
    
with open("syndrome_list.json") as fp:
    SYNDROMES = [d['name'].lower() for d in json.load(fp)]

# add a pipeline to detect syndromes
ruler = nlp.add_pipe("entity_ruler", config={
    "phrase_matcher_attr": "LOWER",
})
# print([{"label": "DISEASE", "pattern": d} for d in nlp.pipe(DISEASES)])
ruler.add_patterns([{"label": "DISEASE", "pattern": d} for d in DISEASES])
ruler.add_patterns([{"label": "SYNDROME", "pattern": s} for s in SYNDROMES])

In [44]:
def parse_text(article):
    html = article['article_text']
    body = Selector(text=html)
    text = ' '.join(s.strip() for s in body.css('#content *::text').getall())
    # break into paragraphs
    for item in body.css('p'):
        yield ' '.join(item.css("*::text").getall())

def get_valid_dates(strings, date_of_article):
    for s in strings:
        result = dateparser.parse(s, settings={"RELATIVE_BASE": date_of_article})
        if result:
            yield result

In [49]:
def parse_reports(paragraphs, date_of_article):
    docs = nlp.pipe(paragraphs)
    for doc in docs:
        with_ent = lambda x: [ent for ent in doc.ents if ent.label_ == x]

        diseases = with_ent("DISEASE")
        syndromes = with_ent("SYNDROME")
        dates = with_ent("DATE")
        locations = with_ent("GPE") # countries, cities and states
#         number = with_ent("CARDINAL")

        if (any(diseases) or any(syndromes)) and any(dates) and any(locations):
            spacy.displacy.render(doc, style="ent")
            for date in get_valid_dates((ent.text for ent in dates), date_of_article):
                report = {
                    'diseases': [ent.text for ent in diseases],
                    'syndromes': [ent.text for ent in syndromes],
                    'locations': [ent.text for ent in locations],
                    'event_date': date
                }
                print(report)
    
for i in range(100):
    date = dateparser.parse(articles[i]['date_of_publication'].replace(" xx:xx:xx", ""))
    print(articles[i]['url'], date)
    parse_reports(parse_text(articles[i]), date)
    

/news-perspective/2022/03/chinas-omicron-covid-19-surge-gains-steam 2022-03-14 00:00:00
/news-perspective/2022/03/news-scan-mar-14-2022 2022-03-14 00:00:00


/home/math2001/.local/lib/python3.8/site-packages/dateparser/date_parser.py:35: PytzUsageWarning: The localize method is no longer necessary, as this time zone supports the fold attribute (PEP 495). For more details on migrating to a PEP 495-compliant implementation, see https://pytz-deprecation-shim.readthedocs.io/en/latest/migration.html
  date_obj = stz.localize(date_obj)
/home/math2001/.local/lib/python3.8/site-packages/dateparser/date_parser.py:35: PytzUsageWarning: The localize method is no longer necessary, as this time zone supports the fold attribute (PEP 495). For more details on migrating to a PEP 495-compliant implementation, see https://pytz-deprecation-shim.readthedocs.io/en/latest/migration.html
  date_obj = stz.localize(date_obj)


/home/math2001/.local/lib/python3.8/site-packages/dateparser/freshness_date_parser.py:76: PytzUsageWarning: The localize method is no longer necessary, as this time zone supports the fold attribute (PEP 495). For more details on migrating to a PEP 495-compliant implementation, see https://pytz-deprecation-shim.readthedocs.io/en/latest/migration.html
  now = self.get_local_tz().localize(now)
/home/math2001/.local/lib/python3.8/site-packages/dateparser/date_parser.py:35: PytzUsageWarning: The localize method is no longer necessary, as this time zone supports the fold attribute (PEP 495). For more details on migrating to a PEP 495-compliant implementation, see https://pytz-deprecation-shim.readthedocs.io/en/latest/migration.html
  date_obj = stz.localize(date_obj)


{'diseases': ['COVID-19'], 'syndromes': [], 'locations': ['South Korea'], 'event_date': datetime.datetime(2021, 3, 14, 0, 0)}
{'diseases': ['COVID-19'], 'syndromes': [], 'locations': ['South Korea'], 'event_date': datetime.datetime(2022, 2, 28, 0, 0)}


/home/math2001/.local/lib/python3.8/site-packages/dateparser/date_parser.py:35: PytzUsageWarning: The localize method is no longer necessary, as this time zone supports the fold attribute (PEP 495). For more details on migrating to a PEP 495-compliant implementation, see https://pytz-deprecation-shim.readthedocs.io/en/latest/migration.html
  date_obj = stz.localize(date_obj)
/home/math2001/.local/lib/python3.8/site-packages/dateparser/date_parser.py:35: PytzUsageWarning: The localize method is no longer necessary, as this time zone supports the fold attribute (PEP 495). For more details on migrating to a PEP 495-compliant implementation, see https://pytz-deprecation-shim.readthedocs.io/en/latest/migration.html
  date_obj = stz.localize(date_obj)
/home/math2001/.local/lib/python3.8/site-packages/dateparser/date_parser.py:35: PytzUsageWarning: The localize method is no longer necessary, as this time zone supports the fold attribute (PEP 495). For more details on migrating to a PEP 495-co

...

/news-perspective/2022/02/news-scan-feb-11-2022 2022-02-11 00:00:00


{'diseases': ['measles'], 'syndromes': [], 'locations': ['Afghanistan'], 'event_date': datetime.datetime(2022, 2, 10, 0, 0)}


/home/math2001/.local/lib/python3.8/site-packages/dateparser/freshness_date_parser.py:76: PytzUsageWarning: The localize method is no longer necessary, as this time zone supports the fold attribute (PEP 495). For more details on migrating to a PEP 495-compliant implementation, see https://pytz-deprecation-shim.readthedocs.io/en/latest/migration.html
  now = self.get_local_tz().localize(now)


{'diseases': ['measles'], 'syndromes': [], 'locations': ['Balkh', 'Ghazni', 'Helmand', 'Kandahar', 'Kabul', 'Balk', 'Ghor', 'Kandahar'], 'event_date': datetime.datetime(2021, 12, 11, 0, 0)}


/home/math2001/.local/lib/python3.8/site-packages/dateparser/freshness_date_parser.py:76: PytzUsageWarning: The localize method is no longer necessary, as this time zone supports the fold attribute (PEP 495). For more details on migrating to a PEP 495-compliant implementation, see https://pytz-deprecation-shim.readthedocs.io/en/latest/migration.html
  now = self.get_local_tz().localize(now)
/home/math2001/.local/lib/python3.8/site-packages/dateparser/date_parser.py:35: PytzUsageWarning: The localize method is no longer necessary, as this time zone supports the fold attribute (PEP 495). For more details on migrating to a PEP 495-compliant implementation, see https://pytz-deprecation-shim.readthedocs.io/en/latest/migration.html
  date_obj = stz.localize(date_obj)
